
`Shapespyer` is a Python toolkit that builds starting configurations for molecular dynamics (MD) simulations of soft matter, such as micelles, vesicles and bilayers in solvent, and equilibrates the systems with GROMACS. We're developing a desktop app `EasyShapes` that uses `shapes.project`, a Shapespyer package, as its backend for building and equilibrating such systems. The notebooks for our section today use `shapes.project` directly. It provides pythonic API that could be used in jupyter, python scripts and GUI, but the same functionality (and more) is available with Shapespyer's command-line tools (`smiles`, `shape`, `gmx-*`). While the command-line tools suit batch runs from shell scripts and cluster job queues, `shapes.project` suits interactive work.

#### Section program following the `EasyShapes` workflow and `shapes.project` layout

The notebooks of this workshop section follow the layout of `shapes.project`. In order to equilibrate a system, a project needs a sample model. A sample model, in turn, needs a structure and a buffer, and a structure needs components with their counter-ions. The notebooks build this up in the opposite direction, one step at a time, from left to right in the diagram:

```
┌─ Library ────────────────────────────────────────────────────────────────────────────────┐
│ Assets downloaded from Shapespyer repo or saved by the user                              │
└─────────────┬──▲───────────────────┬──▲────────────────────────┬──▲───────────────────┬──┘
          load│  │save           load│  │save                load│  │save           load│
              ▼  │                   ▼  │                        ▼  │                   ▼
┌─ Component ──────┐   ╔═ Structure ═════════╗   ┌─ SampleModel ──────┐   ┌─ Project ──────┐
│ .paths           │──►║ .components         ║──►│ .structure         │──►│ .sample_model  │
│ .counter_ion     │   ║ .layers (Ball)      ║   │ .buffer            │   │ .analysis      │
│ .mint            │   ║ .lamellae (Vesicle) ║   │                    │   │                │
│ .mext            │   ║ .motif (Lattice)    ║   │                    │   │                │
└──────────────────┘   ╚═════════════════════╝   └────────────────────┘   └────────────────┘
  notebook 1             notebook 2                notebook 3               notebook 4
└─────────────────── EasyShapes: Sample Model tab ────────────────────┘   Project + Analysis tabs
```

0. **Library** (all notebooks) is represented by the class `Library`, which manages the asset files, both the ready-made molecules and equilibration settings downloaded on first use and the assets you save. Every class that holds files has `load(name)` and `save()`, which use the `Library` object underneath.
1. **Component** (notebook 1) is represented by the classes `Lipid` and `Surfactant`. Components are loaded from the library or created from a structure file or a SMILES string.
2. → **Structure** (notebook 2) is represented by the classes `Ring`, `Rod`, `Bilayer`, `Monolayer`, `Ball`, `Vesicle` and `Lattice`, each arranging copies of components in its shape.
3. **SampleModel** (notebook 3) is a structure in a buffer of solvent and salt, solvated with GROMACS. The buffer is represented by the classes `Buffer`, `Solvent` and `Salt`.
4. **Project** (notebook 4) holds a sample model and its analysis, the staged GROMACS equilibration that relaxes the model.

# 2. Structures

This notebook works with the structures, their layers, and the components they are built from.

```
     ┌───────────┐   ╔═ Structure ═════════╗   ┌─ SampleModel ┐
… ──►│ Component │──►║ .components         ║──►│ .structure   │──► …
     └───────────┘   ║ .layers (Ball)      ║   └──────────────┘
                     ║ .lamellae (Vesicle) ║
                     ║ .motif (Lattice)    ║
                     ╚═════════════════════╝
```

A structure places copies of its components in a shape: a ring, a rod, a bilayer, a ball, a vesicle, or a lattice of other structures.

For each structure in this notebook we:

1. set it up
2. call `.assemble()`, which writes its `.gro` file (no GROMACS needed)
3. call `.export_to()`, which copies the files to a folder you choose
4. look at the result

You can open the exports in VMD or PyMOL.

We have to start with a few setup cells. Make sure Shapespyer is installed in your virtual environment. [Shapespyer installation docs](https://simnavi.gitlab.io/shapespyer.io/installation.html)

In [ ]:
# Setup: check that shapespyer.yaml is next to the notebook
from pathlib import Path

CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files
if not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

The next cell provides helper functions that only draw the structures, or a slice through them, and list the atoms of a component.
These helper functions, as well as the helpers in other notebooks, **are not part of the Shapespyer workflow**. They are all in the `Helpers` class, and the notebook always calls them from it, as `Helpers.<name>()`, so you can tell them apart from the Shapespyer package.

In [ ]:
# Helpers (not part of the Shapespyer workflow)
import mdtraj
import nglview
import numpy as np


class Helpers:
    """Workshop helpers, not part of Shapespyer: call them as Helpers.<name>()."""

    @staticmethod
    def show_structure(structure, box=False):
        """Draw the .gro of an assembled structure; box=True also draws the box of the .gro."""
        gro = next(path for path in structure.paths if path.suffix == ".gro")
        view = nglview.show_structure_file(str(gro))
        if box:
            view.add_representation("unitcell")
        return view

    @staticmethod
    def show_slice(structure, thickness=2.0, box=False):
        """Draw the molecules of an assembled structure whose centre lies in a slab through its middle.

        The slab is thickness nm wide along Z, so a ball or a vesicle shows its cross-section.
        box=True also draws the box of the .gro.
        """
        gro = next(path for path in structure.paths if path.suffix == ".gro")
        frame = mdtraj.load(str(gro))
        z = frame.xyz[0, :, 2]
        residue = np.array([atom.residue.index for atom in frame.top.atoms])
        centres = np.bincount(residue, weights=z) / np.bincount(residue)
        in_slab = np.abs(centres - (z.min() + z.max()) / 2) < thickness / 2
        print(f"{in_slab.sum()} of the {len(centres)} molecules lie in the {thickness} nm slice")
        view = nglview.show_mdtraj(frame.atom_slice(np.flatnonzero(in_slab[residue])))
        if box:
            view.add_representation("unitcell")
        return view

    @staticmethod
    def list_atoms(component):
        """Print the index and the name of each atom of a component, hydrogens left out."""
        frame = mdtraj.load(str(component.structure_file_path))
        atoms = [f"{atom.index} {atom.name}" for atom in frame.top.atoms if atom.element.symbol != "H"]
        print(f"{component.name}, {frame.n_atoms} atoms:", ", ".join(atoms))

In the setup we pass the `shapespyer.yaml` config file, which says where Shapespyer keeps its files. Here it keeps the workshop's files apart from your own Shapespyer data. In the common scenario, users won't have to use the config file, as the `shapes.project` package manages files automatically. Use the config if you want manual control over where the files are kept.

In [ ]:
# Config file, log level and work folder
from shapes.basics.utils import LogConfiguration
from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Workshop files:", config.app_path)

# the level applies to Shapespyer's own handlers; stop the messages there, so no other handler prints them again
LogConfiguration("WARNING").logger.propagate = False

# the exports go to a work folder, which the clean up deletes
WORK = config.app_path / "work"
EXPORTS = WORK / "structure_exports"

In [ ]:
from shapes.basics.defaults import Fill
from shapes.basics.options import ShapeType
from shapes.project import Ball, Bilayer, Lattice, Library, Lipid, Ring, Rod, Surfactant, Vesicle

### The library

Here you choose where the library keeps its files: both the assets downloaded from the Shapespyer assets and the components you save yourself. We give `LocalAssets/` in the application directory, which is also the default. Only the first `Library()` of a session sets the path: later calls return the same library. `clean_temp()` deletes the working files that earlier runs left in Temp.

In [ ]:
library = Library(path=config.app_path / "LocalAssets")
library.clean_temp()

### Ring

The simplest structure: copies of a component on one ring. The same with the command-line tool:
`shape --dio=inp,usr-out -i SDS-single1.gro -o config1 -x .gro -s ring -r SDS -l 15`

Every structure holds:

- `name`: the name of the structure, also the name of its `.gro` file.
- `components`: the components it is built from. `append()` adds one, and `components["SDS"]` gets one by its name.

`assemble()` writes the structure's `.gro` file, and `export_to(folder, overwrite=False)` copies its files into a folder you choose. When you change a setting, call `assemble()` again.

A ring also holds:

- `rmin`: the radius of the empty space in the centre, in nm. We use 2.
- `dmin`: the minimum distance between the "bone" atoms of neighbouring molecules, in nm. We keep the default, 0.5.
- `alpha`: the initial azimuth angle, in the XY plane, of the molecules on the ring, from 0 to 360 degrees. We keep the default, 0.
- `theta`: the initial altitude angle, from the Z axis, of the molecules on the ring, from 0 to 180 degrees. We use 90.
- `fxz`: flattens each molecule in the XZ plane first, so it spreads less along Y.
- `rev`: swaps the interior and the exterior atoms of each molecule.
- `composition`: how much of each component the ring holds. With one component there is nothing to set; notebook 3 sets compositions.

The ring holds about 2π · `rmin` / `dmin` molecules, 25 here. A `rmin` below `dmin` / 2 is raised to it.

Each molecule is placed by its bone: the line from its interior atom `mint` to its exterior atom `mext`, two fields of the component. The structure points the bone away from its centre. In the library's SDS, atom 0 is the S of the head and atom 41 the last H of the tail. The library sets neither, so both are 0, which means unset: the bone then runs from the last atom, the interior one, to the first atom, the exterior one. `mint` and `mext` attributes will be explained further, for now just focus on the ring attributes.

Try:

- `rmin` from 1 to 4 nm, and see the number of molecules follow
- `theta` 0 or 45 instead of 90
- `fxz=True`, or `rev=True` to turn the molecules round

In [ ]:
ring = Ring(
    name="SDS_ring",
    rmin=2.0,
    dmin=0.5,
    alpha=0.0,
    theta=90.0,
    fxz=False,
    rev=False,
)
ring.components.append(Surfactant.load("SDS"))

ring.assemble()
ring.export_to(EXPORTS / "ring", overwrite=True)
Helpers.show_structure(ring)

### Several components: matching `mint` and `mext`

The ring above holds one component. A structure can hold several, and then each molecule is placed by its own bone:

- the bone runs from the interior atom `mint` to the exterior atom `mext`;
- the structure turns each molecule so its bone points away from the centre, with the interior atom on the inside, at about `rmin`;
- so the molecules of different components match when `mint` is the same end of every molecule, and `mext` the other end. The tail ends as `mint` and the polar heads as `mext` put every head outside.

`Helpers.list_atoms()` prints the index and the name of each atom of a component, without the hydrogens, to find the atoms to use.

**Task:** the next cells load POPC and cholesterol (`CHL`) from the library and build a ring of both, with `mint` and `mext` unset. Look at the atoms of both molecules, then set `mint` and `mext` of both components so the molecules match: both heads outside, both tails inside. The solution is in the hidden cell after the ring.

In [ ]:
popc = Lipid.load("POPC")
chl = Lipid.load("CHL")
Helpers.list_atoms(popc)
Helpers.list_atoms(chl)

In [ ]:
mixed_ring = Ring(
    name="POPC_CHL_ring",
    rmin=2.0,
    dmin=0.8,
    alpha=0.0,
    theta=90.0,
    fxz=False,
    rev=False,
)
mixed_ring.components.append(popc)
mixed_ring.components.append(chl)
# the task: set mint and mext of both components, so the molecules match
mixed_ring.components["POPC"].mint = 0
mixed_ring.components["POPC"].mext = 0
mixed_ring.components["CHL"].mint = 0
mixed_ring.components["CHL"].mext = 0

mixed_ring.assemble()
mixed_ring.export_to(EXPORTS / "popc_chl_ring", overwrite=True)
Helpers.show_structure(mixed_ring)

In [ ]:
# Solution: tail ends inside, heads outside
mixed_ring.components["POPC"].mint = 87  # C218, the end of the oleoyl tail
mixed_ring.components["POPC"].mext = 0  # N, in the choline of the head
mixed_ring.components["CHL"].mint = 70  # C27, the end of the side chain
mixed_ring.components["CHL"].mext = 2  # O3, the hydroxyl

mixed_ring.assemble()
mixed_ring.export_to(EXPORTS / "popc_chl_ring", overwrite=True)
Helpers.show_structure(mixed_ring)

### Rod

A stack of rings, or a band spiral. The same with the command-line tool:
`shape --dio=usr-out -i config_SO4C12_smiles.gro -o configS -x .gro -s rod -r SDS -l 15 -t 10`

A rod holds `name`, `components`, `rmin`, `dmin`, `fxz`, `rev` and `composition`, as the ring does, and:

- `turns`: the number of full turns of the rod, 0 or more. More turns make a longer rod. We use 24.

We use `rmin` 1.23 nm and `dmin` 0.4 nm, so about 19 molecules go round each turn. `mint` and `mext` stay unset, as in the ring.

Try:

- `turns` from 5 to 40
- `rmin` and `dmin` as for the ring

In [ ]:
rod = Rod(
    name="SDS_rod",
    rmin=1.23,
    dmin=0.4,
    turns=24,
    fxz=False,
    rev=False,
)
rod.components.append(Surfactant.load("SDS"))
rod.components["SDS"].mint = 0
rod.components["SDS"].mext = 0

rod.assemble()
rod.export_to(EXPORTS / "rod", overwrite=True)
Helpers.show_structure(rod)

### Bilayer

`Monolayer` takes the same fields and builds a single leaflet; it isn't shown here.

A bilayer holds `name`, `components`, `dmin` and `composition`, as the ring does, and:

- `nside`: the number of molecules on each side of the bilayer. We keep the default, 10.
- `zsep`: the minimum distance along Z between the molecules of the two leaflets, in nm. We keep the default, 0.

The middle of the bilayer is its centre, so `mint = 41` puts the tails there, and `mext = 0` the heads outside.

Try:

- `nside` from 5 to 20
- `dmin` from 0.4 to 0.8 nm
- `zsep` from 0.2 to 0.5 nm

In [ ]:
bilayer = Bilayer(
    name="SDS_bilayer",
    dmin=0.5,
    zsep=0.0,
    nside=10,
)
bilayer.components.append(Surfactant.load("SDS"))
bilayer.components["SDS"].mint = 41
bilayer.components["SDS"].mext = 0

bilayer.assemble()
bilayer.export_to(EXPORTS / "bilayer", overwrite=True)
Helpers.show_structure(bilayer)

### Ball

An SDS micelle: molecules on rings, or on a spiral, round a sphere. The same with the command-line tool:
`shape --dio=inp,usr-out -i config_SO4C12_smiles.gro -o configS -x .gro -s ball -r SDS --mint=41 -l 16 --dmin=0.4`

A ball holds `name`, `components`, `fxz` and `rev`, as the ring does, and:

- `fill`: how the molecules are placed on the sphere, a `Fill`: on rings (`Fill.RINGS0`, the default, or `Fill.RINGS`) or on a Fibonacci spiral (`Fill.FIBO`).
- `layers`: the concentric layers of the ball, one to start with. Each layer holds its own `rmin`, `dmin` and `composition`, as the ring does. `layers.add()` adds a layer, with the `dmin` of the layer before and an `rmin` just outside it. A layer's `rmin` must be at least 1.6 times its `dmin`.

We use one layer with `rmin` 1.3 nm and `dmin` 0.4 nm. In a micelle the tails go inside: `mint = 41` makes the last H of the tail the interior atom, and `mext = 0` the S of the head the exterior one.

From here on, `Helpers.show_slice()` draws only the molecules in a slab through the middle of the structure, `thickness` nm wide along Z (2 by default), so you see the inside. `Helpers.show_structure()` draws the whole structure.

Try:

- `fill=Fill.FIBO`
- `rmin` from 1 to 2 nm and `dmin` from 0.3 to 0.6 nm: a larger `rmin` or a smaller `dmin` puts more molecules in
- `mint = 0` and `mext = 41`, to put the heads inside

In [ ]:
ball = Ball(
    name="Ball_1",
    fill=Fill.RINGS0,
    fxz=False,
    rev=False,
)
ball.components.append(Surfactant.load("SDS"))
ball.layers[0].rmin = 1.3
ball.layers[0].dmin = 0.4
ball.components["SDS"].mint = 41
ball.components["SDS"].mext = 0

ball.assemble()
ball.export_to(EXPORTS / "ball1", overwrite=True)
Helpers.show_slice(ball)

### Ball with several layers

A ball can hold several concentric layers, each with its own `rmin`, `dmin` and `composition`:

- `ball.layers.add()` adds a layer outside the last one, and `ball.layers.remove(1)` removes the second one. A new layer takes the `dmin` of the layer before and an `rmin` just outside it; here we set both.
- Every component of the ball joins every layer, present, with the molar ratio 1. A layer's `composition` changes that: `set_mol_ratio(name, value)` sets the molar ratio of a component in that layer, and `set_presence(name, False)` leaves it out. Notebook 3 explains compositions in detail.

Here a ball of POPC and cholesterol, with `mint` and `mext` as in the task above: the inner layer holds POPC and cholesterol at 2 : 1, the outer layer POPC only.

Try:

- `set_presence("CHL", True)` in the outer layer
- remove the outer layer, `layered_ball.layers.remove(1)`, and assemble again

In [ ]:
layered_ball = Ball(
    name="POPC_CHL_ball",
    fill=Fill.RINGS0,
    fxz=False,
    rev=False,
)
layered_ball.components.append(Lipid.load("POPC"))
layered_ball.components.append(Lipid.load("CHL"))
layered_ball.components["POPC"].mint = 87
layered_ball.components["POPC"].mext = 0
layered_ball.components["CHL"].mint = 70
layered_ball.components["CHL"].mext = 2

layered_ball.layers[0].rmin = 1.5
layered_ball.layers[0].dmin = 0.8
layered_ball.layers[0].composition.set_mol_ratio("POPC", 2.0)
layered_ball.layers[0].composition.set_mol_ratio("CHL", 1.0)

layered_ball.layers.add()
layered_ball.layers[1].rmin = 5.5
layered_ball.layers[1].dmin = 1.0
layered_ball.layers[1].composition.set_presence("CHL", False)

layered_ball.assemble()
layered_ball.export_to(EXPORTS / "layered_ball", overwrite=True)
Helpers.show_slice(layered_ball)

### Vesicle

A vesicle is made of lamellae, each with an inner and an outer leaflet. The same with the command-line tool:
`shape --dio=inp,usr-out -i config_SO4C12_smiles.gro -o configS -x .gro -s ves -l 16 -r SDS --mint=41 --rev`

A vesicle holds `name`, `components`, `fill`, `fxz` and `rev`, as the ball does, and:

- `lamellae`: the concentric lamellae, one to start with; `lamellae.add()` adds one. A lamella holds:
  - `rmin`: the radius of the empty space in the centre, where the inner leaflet starts, in nm.
  - `inner_leaflet_dmin` and `outer_leaflet_dmin`: the minimum distance between the bone atoms in each leaflet, in nm, 0.5 by default.
  - `shell_thickness`, `symmetric` and the compositions of the leaflets: notebook 3 sets them, for an asymmetric vesicle.

We use `rmin` 1.23 nm, `inner_leaflet_dmin` 0.4 nm and the Fibonacci fill, with `mint` and `mext` as in the ball. The slice shows the inside of the lamella, as for the ball.

Try:

- `rmin` from 1 to 5 nm
- `outer_leaflet_dmin` 0.6 nm
- `fill=Fill.RINGS0`

In [ ]:
vesicle = Vesicle(
    name="SDS_vesicle",
    fill=Fill.FIBO,
    fxz=False,
    rev=False,
)
vesicle.components.append(Surfactant.load("SDS"))
vesicle.lamellae[0].rmin = 1.23
vesicle.lamellae[0].inner_leaflet_dmin = 0.4
vesicle.components["SDS"].mint = 41
vesicle.components["SDS"].mext = 0

vesicle.assemble()
vesicle.export_to(EXPORTS / "vesicle", overwrite=True)
Helpers.show_slice(vesicle)

### Vesicle with several lamellae

A vesicle can hold several concentric lamellae, as a ball holds layers:

- `vesicle.lamellae.add()` adds a lamella outside the last one, and `vesicle.lamellae.remove(1)` removes the second one.
- A new lamella takes the leaflet `dmin`s of the lamella before and an `rmin` just outside it; here we set them. Set `rmin` on every lamella, or to `None` on all but the first, to have the generator scale the rest.
- The two leaflets of a lamella share one composition while `symmetric` is `True`. With `symmetric = False`, each leaflet has its own, `inner_leaflet_composition` and `outer_leaflet_composition`, set as in the layers of the ball; `swap()` exchanges them. Notebook 3 builds an asymmetric vesicle in detail.

Here a vesicle of POPC and cholesterol in two lamellae, with `mint` and `mext` as in the task above. The inner lamella holds both at 1 : 1 in both leaflets, the defaults. The outer lamella holds POPC only in its inner leaflet, and POPC and cholesterol at 3 : 1 in its outer one.

Try:

- `outer_lamella.swap()`
- remove the outer lamella, `layered_vesicle.lamellae.remove(1)`, and assemble again

In [ ]:
layered_vesicle = Vesicle(
    name="POPC_CHL_vesicle",
    fill=Fill.FIBO,
    fxz=False,
    rev=False,
)
layered_vesicle.components.append(Lipid.load("POPC"))
layered_vesicle.components.append(Lipid.load("CHL"))
layered_vesicle.components["POPC"].mint = 87
layered_vesicle.components["POPC"].mext = 0
layered_vesicle.components["CHL"].mint = 70
layered_vesicle.components["CHL"].mext = 2

inner_lamella = layered_vesicle.lamellae[0]
inner_lamella.rmin = 3.0
inner_lamella.shell_thickness = 5.0
inner_lamella.inner_leaflet_dmin = 1.2
inner_lamella.outer_leaflet_dmin = 1.5

layered_vesicle.lamellae.add()
outer_lamella = layered_vesicle.lamellae[1]
outer_lamella.rmin = 9.5
outer_lamella.shell_thickness = 5.0
outer_lamella.inner_leaflet_dmin = 1.8
outer_lamella.outer_leaflet_dmin = 2.0
outer_lamella.symmetric = False
outer_lamella.inner_leaflet_composition.set_presence("CHL", False)
outer_lamella.outer_leaflet_composition.set_mol_ratio("POPC", 3.0)
outer_lamella.outer_leaflet_composition.set_mol_ratio("CHL", 1.0)

layered_vesicle.assemble()
layered_vesicle.export_to(EXPORTS / "layered_vesicle", overwrite=True)
Helpers.show_slice(layered_vesicle)

### Lattice

A lattice repeats a motif structure on its nodes, here an SDS ball on a simple cubic lattice. The lattice builds its motif itself, so the motif isn't assembled here. `box=True` makes the helper also draw the box of the `.gro` file; it works for every structure.

A lattice holds `name`, as every structure does, and:

- `motif`: the structure repeated on the lattice nodes: a ring, a rod, a bilayer, a ball or a vesicle.
- `lattice_type`: a `ShapeType`: `ShapeType.LCUB` (simple cubic), `ShapeType.LBCC` (body-centred cubic), `ShapeType.LFCC` (face-centred cubic) or `ShapeType.LHCP` (hexagonal close packing).
- `nlatx`, `nlaty` and `nlatz`: the number of lattice unit cells repeated along X, Y and Z. We use 2 × 1 × 1.
- `sbuff`: the solvation buffer around the generated structure, in nm. We keep the default, 1.
- `alpha` and `theta`: the initial angles of the molecules, as in the ring. We keep the defaults, 0.

Try:

- `lattice_type=ShapeType.LBCC` or `ShapeType.LFCC`
- `nlatx`, `nlaty` and `nlatz` up to 3; more cells hold more motifs, and take longer
- a ring as the motif

In [ ]:
motif = Ball(
    name="SDS_motif",
    fill=Fill.RINGS0,
    fxz=False,
    rev=False,
)
motif.components.append(Surfactant.load("SDS"))
motif.layers[0].rmin = 1.3
motif.layers[0].dmin = 0.4

lattice = Lattice(
    name="SDS_lattice",
    motif=motif,
    lattice_type=ShapeType.LCUB,
    nlatx=2,
    nlaty=1,
    nlatz=1,
    sbuff=1.0,
    alpha=0.0,
    theta=0.0,
)

lattice.assemble()
lattice.export_to(EXPORTS / "lattice", overwrite=True)
Helpers.show_structure(lattice, box=True)

### Clean up (optional)

This deletes the work folder, with the exports, and Temp, where the structures were assembled. `force=True` is needed for Temp, because the structures of this session still hold their files there.

In [ ]:
import shutil

shutil.rmtree(WORK, ignore_errors=True)
library.clean_temp(force=True)
print(WORK.exists())